## Heart Disease Detection

#### Loading Data

In [194]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"heart_patients_dirty_dataset.csv")
df.head()

,Patient_ID,Age,Gender,Chest_Pain_Type,Resting_BP_mmHg,Cholesterol_mg/dl,Max_Heart_Rate,ST_Depression,Exercise_Induced_Angina,Heart_Disease
0,524.0,87.0,NaN,NaN,151.343081,162.378238,162.287641,2.801430,Yes,zero
1,603.0,66.0,male,NaN,101.214098,218.380992,155.846201,0.182318,No,zero
2,527.0,87.0,FEMALE,Atypical Angina,150.354093,248.765623,195.335192,0.875113,yes,1
3,32.0,87.0,FEMALE,Non-Anginal Pain,124.865389,NaN,180.730370,0.300505,yes,NaN
4,617.0,45.0,NaN,Typical Angina,112.437303,211.572296,121.968906,NaN,No,one


#### making column names lowercase for easier analysis

In [195]:
df.columns = df.columns.str.lower()

In [196]:
df.columns

Index(['patient_id', 'age', 'gender', 'chest_pain_type', 'resting_bp_mmhg',
       'cholesterol_mg/dl', 'max_heart_rate', 'st_depression',
       'exercise_induced_angina', 'heart_disease'],
      dtype='object')

#### very messy dataset, we need to clean it first

#### lets start with filling null values

In [197]:
df['patient_id'].isnull().sum()

np.int64(15)

#### 15 id's missing, let's reassign this column

In [198]:
df["patient_id"] = range(1, len(df) + 1)

In [199]:
df["patient_id"]

0          1
1          2
2          3
3          4
4          5
        ... 
1015    1016
1016    1017
1017    1018
1018    1019
1019    1020
Name: patient_id, Length: 1020, dtype: int64

In [200]:
df['age'].isnull().sum()

np.int64(15)

In [201]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")

In [202]:
df["age"] = df["age"].fillna(df["age"].median())

In [203]:
df['gender'].isnull().sum()

np.int64(216)

In [204]:
df['gender'] = df['gender'].str.lower()

In [205]:
df['gender']

0          NaN
1         male
2       female
3       female
4          NaN
         ...  
1015      male
1016    female
1017       NaN
1018    female
1019    female
Name: gender, Length: 1020, dtype: object

#### Let's do mapping, it's vital for EDA analysis

In [206]:
if not df["gender"].mode().empty:
    df["gender"].fillna(df["gender"].mode()[0], inplace=True)
else:
    print("Mode is empty — check data!")

In [207]:
df['gender']

0         male
1         male
2       female
3       female
4         male
         ...  
1015      male
1016    female
1017      male
1018    female
1019    female
Name: gender, Length: 1020, dtype: object

#### i have filled the missing gender values with the most frequent values

In [208]:
df['gender'].isnull().sum()

np.int64(0)

#### Encoding gender with Male=1 Female=0

In [209]:
df["gender"] = df["gender"].map({
    "male": 1,
    "female": 0
})

In [210]:
df['gender']

0       1
1       1
2       0
3       0
4       1
       ..
1015    1
1016    0
1017    1
1018    0
1019    0
Name: gender, Length: 1020, dtype: int64

In [211]:
df['chest_pain_type'] = df['chest_pain_type'].str.lower()

#### mapping chest pain type

In [212]:
df["chest_pain_type"] = df["chest_pain_type"].map({
    "typical angina": 0,
    "atypical angina" : 1,
    "non-anginal pain": 2,
    "asymptomatic": 3
})

In [213]:
missing_pct = df["chest_pain_type"].isnull().mean() * 100
print(missing_pct)

21.764705882352942


In [214]:
df["chest_pain_type"] = df["chest_pain_type"].fillna(-1)

#### filled missing values with "Unknown: -1"

In [215]:
df['resting_bp_mmhg'].isnull().sum()

np.int64(16)

#### filling several numerical values using median

In [216]:
df["resting_bp_mmhg"] = df["resting_bp_mmhg"].fillna(df["resting_bp_mmhg"].median())

In [217]:
df['cholesterol_mg/dl'].isnull().sum()

np.int64(15)

In [218]:
df["cholesterol_mg/dl"] = df["cholesterol_mg/dl"].fillna(df["cholesterol_mg/dl"].median())

In [219]:
df['max_heart_rate'].isnull().sum()

np.int64(16)

In [220]:
df["max_heart_rate"] = df["max_heart_rate"].fillna(df["max_heart_rate"].median())

In [221]:
df['st_depression'].isnull().sum()

np.int64(25)

In [222]:
df["st_depression"] = df["st_depression"].replace("N/A", np.nan)

In [223]:
df['st_depression'].isnull().sum()

np.int64(25)

In [224]:
df["st_depression"] = df["st_depression"].fillna(df["st_depression"].median())

In [225]:
df['exercise_induced_angina'].isnull().sum()

np.int64(219)

In [226]:
df['exercise_induced_angina'].str.lower()

0       yes
1        no
2       yes
3       yes
4        no
       ... 
1015    NaN
1016    yes
1017     no
1018    NaN
1019    yes
Name: exercise_induced_angina, Length: 1020, dtype: object

In [227]:
df["exercise_induced_angina"] = df["exercise_induced_angina"].map({
    "yes": 1,
    "no" : 0
})

In [228]:
df["exercise_induced_angina"] = df["exercise_induced_angina"].fillna(-1)

In [229]:
df["heart_disease"] = df["heart_disease"].str.lower()

In [230]:
df["heart_disease"] = df["heart_disease"].replace({
    "zero": 0,
    "one": 1
})

In [231]:
df['heart_disease'].isnull().sum()

np.int64(238)

#### 238(23%) missing results, as it is a vital point for our overall analysis, we are droping the rows with missing heart disease result

In [232]:
df = df.dropna(subset=["heart_disease"])

In [233]:
df.shape

(782, 10)

#### enough data remains for analysis

In [234]:
df.duplicated().sum()

np.int64(0)

In [235]:
df["patient_id"] = pd.to_numeric(df["patient_id"], errors="coerce").astype("Int64")
df["age"] = pd.to_numeric(df["age"], errors="coerce").astype("Int64")
df["gender"] = pd.to_numeric(df["gender"], errors="coerce").astype("Int64")
df["chest_pain_type"] = pd.to_numeric(df["chest_pain_type"], errors="coerce").astype("Int64")

df["resting_bp_mmhg"] = pd.to_numeric(df["resting_bp_mmhg"], errors="coerce")

df["cholesterol_mg/dl"] = pd.to_numeric(df["cholesterol_mg/dl"], errors="coerce")
df["max_heart_rate"] = pd.to_numeric(df["max_heart_rate"], errors="coerce")
df["st_depression"] = pd.to_numeric(df["st_depression"], errors="coerce")

df["exercise_induced_angina"] = pd.to_numeric(df["exercise_induced_angina"], errors="coerce").astype("Int64")
df["heart_disease"] = pd.to_numeric(df["heart_disease"], errors="coerce").astype("Int64")

In [236]:
df.head()

,patient_id,age,gender,chest_pain_type,resting_bp_mmhg,cholesterol_mg/dl,max_heart_rate,st_depression,exercise_induced_angina,heart_disease
0,1,87,1,-1,151.343081,162.378238,162.287641,2.801430,-1,0
1,2,66,1,-1,101.214098,218.380992,155.846201,0.182318,-1,0
2,3,87,0,1,150.354093,248.765623,195.335192,0.875113,1,1
4,5,45,1,0,112.437303,211.572296,121.968906,0.982994,-1,1
6,7,89,0,3,130.057209,252.086233,164.887272,1.889154,-1,1


In [237]:
df.describe()

,patient_id,age,gender,chest_pain_type,resting_bp_mmhg,cholesterol_mg/dl,max_heart_rate,st_depression,exercise_induced_angina,heart_disease
count,782.0,782.0,782.0,782.0,782.000000,782.000000,782.000000,782.000000,782.0,782.0
mean,510.945013,60.016624,0.603581,0.97954,121.732230,207.284941,149.663734,0.984445,-0.634271,0.514066
std,294.054618,17.20721,0.489466,1.463894,18.502482,74.943150,26.874293,0.969043,0.773606,0.500122
min,1.0,30.0,0.0,-1.0,75.894170,50.443201,74.809192,-2.170426,-1.0,0.0
25%,257.25,46.0,0.0,0.0,110.377716,167.092753,132.929231,0.350920,-1.0,0.0
50%,509.5,61.0,1.0,1.0,120.328332,201.722051,149.947067,0.982994,-1.0,1.0
75%,762.75,75.0,1.0,2.0,130.567598,234.094600,165.389157,1.636318,-1.0,1.0
max,1020.0,89.0,1.0,3.0,237.905224,854.358432,297.129217,4.076860,1.0,1.0


#### It looks like all values are within range (30<age<89, gender 0 and 1, chest pain type from -1 to 3 etc). No possible outlying unrealistic value ( max resting_bp_mmhg 237, max cholesterol 854 etc). So the dataset is so clean now!

In [238]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 782 entries, 0 to 1019
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   patient_id               782 non-null    Int64  
 1   age                      782 non-null    Int64  
 2   gender                   782 non-null    Int64  
 3   chest_pain_type          782 non-null    Int64  
 4   resting_bp_mmhg          782 non-null    float64
 5   cholesterol_mg/dl        782 non-null    float64
 6   max_heart_rate           782 non-null    float64
 7   st_depression            782 non-null    float64
 8   exercise_induced_angina  782 non-null    Int64  
 9   heart_disease            782 non-null    Int64  
dtypes: Int64(6), float64(4)
memory usage: 71.8 KB


#### All data types are also perfect!

In [239]:
df.to_csv("clean_heart_data.csv", index=False)